# Bloco 05 — Escrita de Dados e Projeto Final

Neste bloco vamos aprender a **gravar dados** em diferentes formatos e modos, e depois aplicar tudo
que aprendemos nos Blocos 01–04 em um **projeto final completo** com o dataset Northwind:

- Formatos de escrita (Delta, Parquet, CSV, JSON)
- Modos de escrita (`overwrite`, `append`, `error`, `ignore`)
- Particionamento na escrita (`partitionBy`)
- `repartition()` vs `coalesce()`
- **Projeto Final** — Pipeline completo: leitura → transformação → relatórios → escrita

## Setup

In [0]:
# pyspark.sql.functions — modulo principal de funcoes do PySpark.
# Contem funcoes para manipulacao de colunas, agregacoes, strings, datas, etc.
# Importamos como "F" por convencao para evitar conflito com built-ins do Python.
from pyspark.sql import functions as F

# pyspark.sql.window.Window — classe para definir janelas (window functions).
# Usada no Projeto Final para calcular receita YTD (acumulada) e segmentacao
# de clientes com ntile(). Sintaxe: Window.partitionBy("grupo").orderBy("ordem")
from pyspark.sql.window import Window

CATALOG = "northwind"
SCHEMA = "bronze"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# Criar Volume para armazenar arquivos (substitui DBFS /tmp)
spark.sql("CREATE VOLUME IF NOT EXISTS demo_files")
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/demo_files"
print(f"Volume disponível em: {VOLUME_PATH}")

orders = spark.table("orders")
order_details = spark.table("order_details")
customers = spark.table("customers")
products = spark.table("products")
categories = spark.table("categories")

---
## 1. Formatos de Escrita

O Spark suporta diversos formatos de saída. Vamos ver os principais:

| Formato  | Uso típico                         | Vantagem principal             |
|----------|------------------------------------|--------------------------------|
| **Delta**   | Tabelas no Lakehouse            | ACID, time travel, otimização  |
| **Parquet** | Data lake / interchange          | Colunar, compacto, rápido      |
| **CSV**     | Exportação / integração simples  | Legível, universal             |
| **JSON**    | APIs / logs                      | Semi-estruturado, flexível     |

### 1.1 Delta (formato preferido no Databricks)

In [0]:
# Criando um DataFrame simples para demonstração
df_demo = spark.createDataFrame(
    [(1, "Alice", 100.0), (2, "Bob", 200.0), (3, "Carol", 150.0)],
    ["id", "nome", "valor"]
)

# Escrita em formato Delta como tabela gerenciada
df_demo.write.format("delta").mode("overwrite").saveAsTable("demo_delta")

# Verificar
spark.table("demo_delta").show()

### 1.2 Parquet (formato colunar)

In [0]:
# Escrita em Parquet
df_demo.write.parquet(f"{VOLUME_PATH}/demo_parquet", mode="overwrite")

# Leitura de volta
spark.read.parquet(f"{VOLUME_PATH}/demo_parquet").show()

### 1.3 CSV (para exportação)

In [0]:
# Escrita em CSV com cabeçalho
df_demo.write.csv(f"{VOLUME_PATH}/demo_csv", header=True, mode="overwrite")

# Leitura de volta
spark.read.csv(f"{VOLUME_PATH}/demo_csv", header=True, inferSchema=True).show()

### 1.4 JSON

In [0]:
# Escrita em JSON
df_demo.write.json(f"{VOLUME_PATH}/demo_json", mode="overwrite")

# Leitura de volta
spark.read.json(f"{VOLUME_PATH}/demo_json").show()

---
## 2. Modos de Escrita

O parâmetro `mode` controla o que acontece quando o destino já existe:

| Modo               | Comportamento                                |
|--------------------|----------------------------------------------|
| `overwrite`        | Substitui os dados existentes                |
| `append`           | Adiciona aos dados existentes                |
| `error` / `errorifexists` | Falha se já existir (padrão)          |
| `ignore`           | Ignora silenciosamente se já existir         |

In [0]:
# Modo overwrite — substitui tudo
df_demo.write.format("delta").mode("overwrite").save(f"{VOLUME_PATH}/modos_escrita")
print("overwrite:", spark.read.format("delta").load(f"{VOLUME_PATH}/modos_escrita").count(), "registros")

# Modo append — adiciona registros
df_demo.write.format("delta").mode("append").save(f"{VOLUME_PATH}/modos_escrita")
print("append:", spark.read.format("delta").load(f"{VOLUME_PATH}/modos_escrita").count(), "registros")

# Modo ignore — não faz nada se já existir
df_demo.write.format("delta").mode("ignore").save(f"{VOLUME_PATH}/modos_escrita")
print("ignore:", spark.read.format("delta").load(f"{VOLUME_PATH}/modos_escrita").count(), "registros")

In [0]:
# Modo error — lança exceção se já existir
try:
    df_demo.write.format("delta").mode("error").save(f"{VOLUME_PATH}/modos_escrita")
except Exception as e:
    print(f"Erro esperado: {type(e).__name__}")
    print(f"  → O modo 'error' falha porque o destino já existe.")

---
## 3. Particionamento na Escrita

O método `partitionBy()` organiza os arquivos em subdiretórios baseados nos valores de uma ou mais colunas.
Isso pode **acelerar muito** as leituras quando filtramos por essas colunas.

**Quando particionar:**
- A coluna de partição tem **cardinalidade baixa** (poucos valores distintos)
- As queries frequentemente filtram por essa coluna
- O dataset é grande o suficiente (> 1 GB)

**Quando NÃO particionar:**
- A coluna tem **alta cardinalidade** (milhares de valores) — gera arquivos pequenos demais
- O dataset é pequeno — overhead maior que o benefício
- Não há padrão claro de filtro nas queries

In [0]:
# Particionar orders por ano
(
    orders
    .withColumn("ano", F.year("order_date"))
    .write.format("delta")
    .partitionBy("ano")
    .mode("overwrite")
    .save(f"{VOLUME_PATH}/orders_partitioned")
)

print("Escrita particionada concluída!")

# Leitura filtrando por partição (Spark faz partition pruning)
df_1997 = (
    spark.read.format("delta")
    .load(f"{VOLUME_PATH}/orders_partitioned")
    .filter(F.col("ano") == 1997)
)
print(f"Pedidos de 1997: {df_1997.count()}")

---
## 4. repartition() vs coalesce()

Antes de escrever, muitas vezes queremos controlar o **número de arquivos** gerados.

| Método          | Shuffle? | Pode aumentar? | Pode diminuir? | Quando usar                        |
|-----------------|----------|----------------|----------------|------------------------------------|  
| `repartition(n)` | Sim     | Sim            | Sim            | Redistribuir dados uniformemente    |
| `coalesce(n)`    | Não     | Não            | Sim            | Reduzir partições sem shuffle       |

**Regra prática:** use `coalesce()` para reduzir (mais eficiente), `repartition()` para aumentar ou redistribuir.

In [0]:
print(f"Partições originais de orders: {orders.rdd.getNumPartitions()}")

# repartition — redistribui com shuffle
df_repart = orders.repartition(8)
print(f"Após repartition(8): {df_repart.rdd.getNumPartitions()}")

# coalesce — reduz sem shuffle
df_coal = orders.coalesce(1)
print(f"Após coalesce(1): {df_coal.rdd.getNumPartitions()}")

In [0]:
# Uso prático: salvar como arquivo único para exportação
(
    orders
    .coalesce(1)
    .write.csv(f"{VOLUME_PATH}/orders_single_file", header=True, mode="overwrite")
)
print("Exportado como arquivo CSV único!")

---
---
## Projeto Final — Pipeline Northwind Completo

Agora vamos construir um **pipeline completo** de **leitura → transformação → escrita**,
aplicando tudo que aprendemos nos Blocos 01–05.

O objetivo é gerar **5 relatórios analíticos** a partir do dataset Northwind e salvá-los como tabelas Delta.

### Etapa 1 — Leitura das Tabelas

In [0]:
orders = spark.table("orders")
order_details = spark.table("order_details")
customers = spark.table("customers")
products = spark.table("products")
categories = spark.table("categories")

print(f"Orders: {orders.count()}, Order Details: {order_details.count()}")
print(f"Customers: {customers.count()}, Products: {products.count()}, Categories: {categories.count()}")

### Etapa 2 — Transformação (Base Analítica)

Vamos construir uma **base analítica unificada** com joins de todas as tabelas e cálculo de receita.

In [0]:
# Base: calcular receita por item
od_receita = order_details.withColumn(
    "receita", F.col("unit_price") * F.col("quantity") * (F.lit(1.0) - F.col("discount"))
)

# Join principal: orders + order_details + customers + products + categories
base = (
    od_receita
    .join(orders, "order_id")
    .join(customers, "customer_id")
    .join(products, od_receita.product_id == products.product_id)
    .join(categories, "category_id")
    .withColumn("ano", F.year("order_date"))
    .withColumn("mes", F.month("order_date"))
    .withColumn("dias_para_entrega",
        F.datediff(F.col("shipped_date"), F.col("order_date")))
    .select(
        orders.order_id, "customer_id", "company_name", "country",
        "order_date", "shipped_date", "ano", "mes", "dias_para_entrega",
        products.product_id, "product_name", "category_name",
        "receita"
    )
)

print(f"Base analítica: {base.count()} linhas")
base.printSchema()

In [0]:
base.show(10, truncate=False)

### Relatório 1 — Receita Mensal com YTD e Crescimento

Receita mensal, acumulado no ano (YTD) e percentual de crescimento mês a mês.

In [0]:
w = Window.partitionBy("ano").orderBy("mes")

relatorio_1 = (
    base.groupBy("ano", "mes")
    .agg(F.round(F.sum("receita"), 2).alias("receita_mensal"))
    .withColumn("receita_ytd", F.round(F.sum("receita_mensal").over(w), 2))
    .withColumn("crescimento_pct",
        F.round(
            (F.col("receita_mensal") - F.lag("receita_mensal").over(w))
            / F.lag("receita_mensal").over(w) * 100, 2
        ))
    .orderBy("ano", "mes")
)

relatorio_1.show(30)

### Relatório 2 — Top 10 Produtos por Receita

In [0]:
relatorio_2 = (
    base.groupBy("product_name")
    .agg(F.round(F.sum("receita"), 2).alias("total_vendas"))
    .orderBy(F.col("total_vendas").desc())
    .limit(10)
)

relatorio_2.show(truncate=False)

### Relatório 3 — Segmentação de Clientes (Quintis)

Dividimos os clientes em 5 grupos usando `ntile(5)`: grupo 1 = maiores compradores, grupo 5 = menores.

In [0]:
receita_cliente = (
    base.groupBy("company_name")
    .agg(F.round(F.sum("receita"), 2).alias("total"))
)

w_ntile = Window.orderBy(F.col("total").desc())

relatorio_3 = receita_cliente.withColumn("grupo", F.ntile(5).over(w_ntile))

relatorio_3.show(20, truncate=False)

### Relatório 4 — Clientes do UK com Receita > 1000

In [0]:
relatorio_4 = (
    base.filter(F.lower(F.col("country")) == "uk")
    .groupBy("company_name")
    .agg(F.round(F.sum("receita"), 2).alias("payments"))
    .filter(F.col("payments") > 1000)
    .orderBy(F.col("payments").desc())
)

relatorio_4.show(truncate=False)

### Relatório 5 — Receita por Categoria × Ano (Pivot)

In [0]:
relatorio_5 = (
    base.groupBy("category_name")
    .pivot("ano")
    .agg(F.round(F.sum("receita"), 2))
    .orderBy("category_name")
)

relatorio_5.show(truncate=False)

### Etapa 3 — Escrita dos Relatórios

Salvamos todos os relatórios como **tabelas Delta** em um schema `northwind_gold` (camada Gold do Medallion).

In [0]:
OUTPUT_SCHEMA = "northwind_gold"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {OUTPUT_SCHEMA}")

relatorio_1.write.format("delta").mode("overwrite").saveAsTable(f"{OUTPUT_SCHEMA}.receita_mensal_ytd")
relatorio_2.write.format("delta").mode("overwrite").saveAsTable(f"{OUTPUT_SCHEMA}.top_10_produtos")
relatorio_3.write.format("delta").mode("overwrite").saveAsTable(f"{OUTPUT_SCHEMA}.segmentacao_clientes")
relatorio_4.write.format("delta").mode("overwrite").saveAsTable(f"{OUTPUT_SCHEMA}.clientes_uk_1000")
relatorio_5.write.format("delta").mode("overwrite").saveAsTable(f"{OUTPUT_SCHEMA}.receita_categoria_ano")

print("Todas as tabelas Delta salvas com sucesso!")

In [0]:
# Exportar base analítica consolidada como CSV
base.coalesce(1).write.csv(f"{VOLUME_PATH}/base_analitica", header=True, mode="overwrite")

print("Base analítica exportada como CSV!")

### Verificação Final

In [0]:
print("Tabelas salvas no schema northwind_gold:")
print("=" * 50)

for table in ["receita_mensal_ytd", "top_10_produtos", "segmentacao_clientes", "clientes_uk_1000", "receita_categoria_ano"]:
    count = spark.table(f"{OUTPUT_SCHEMA}.{table}").count()
    print(f"  {table}: {count} registros")

print("=" * 50)
print("Pipeline completo executado com sucesso!")

---
## Resumo do Bloco 05

Neste bloco aprendemos:

1. **Formatos de escrita** — Delta (preferido), Parquet, CSV, JSON
2. **Modos de escrita** — `overwrite`, `append`, `error`, `ignore`
3. **Particionamento** — `partitionBy()` para organizar arquivos por coluna
4. **Controle de partições** — `repartition()` (com shuffle) vs `coalesce()` (sem shuffle)
5. **Projeto Final** — Pipeline completo com 5 relatórios analíticos salvos como tabelas Delta

### Conceitos aplicados no Projeto Final (Blocos 01–05):
- `spark.table()` para leitura (Bloco 01)
- `withColumn()`, `select()`, `filter()` para transformações (Bloco 02)
- `join()` para cruzamento de tabelas (Bloco 03)
- `groupBy()`, `agg()`, `Window`, `pivot()`, `ntile()` para agregações (Bloco 04)
- `write.format("delta").saveAsTable()` para escrita (Bloco 05)